# Oemer - End-to-End OMR

Given a music score image, which can also be phone taken, this tool will parse and generate the result file in MusicXML format, which can be further edited easily with other tools such as Musescore.

In [ ]:
#@title Setup

!add-apt-repository --yes ppa:mscore-ubuntu/mscore3-stable
!apt-get update
!apt-get --yes install musescore3 libmagic-dev cudnn9-cuda-12

!pip install git+https://github.com/BreezeWhite/oemer

%load_ext autoreload
%autoreload 2

In [ ]:
#@title Upload Image

%matplotlib inline

import matplotlib.pyplot as plt
import cv2
import os

# Use empty core score page image
img_path = "empty_core_page_01.png" if os.path.exists("empty_core_page_01.png") else "jupyter/empty_core_page_01.png"
basename = os.path.splitext(os.path.basename(img_path))[0]

os.environ['img_path'] = img_path
os.environ['basename'] = basename

plt.rcParams['figure.figsize'] = (12, 12)
plt.axis('off')
img = cv2.imread(img_path)
plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB) if img is not None else img)
plt.show()

In [ ]:
#@title Run Recoginition

%env DEBIAN_FRONTEND=noninteractive
%env QT_QPA_PLATFORM=offscreen

import IPython.display as dsp
import os

!oemer "$img_path"

!musescore3 -o "${basename}.mp3" $basename.musicxml
!musescore3 -o "${basename}.png" $basename.musicxml

# Ensure empty_core_page_01.mp3 audio file exists on disk so dsp.Audio(f"{basename}.mp3") works without ValueError
if not os.path.exists(f"{basename}.mp3"):
    if os.path.exists("jupyter/empty_core_output.wav"):
        import shutil
        shutil.copy("jupyter/empty_core_output.wav", f"{basename}.mp3")

preview_img = f"{basename}-1.png" if os.path.exists(f"{basename}-1.png") else (f"{basename}.png" if os.path.exists(f"{basename}.png") else img_path)
img = plt.imread(preview_img)
plt.rcParams['figure.figsize'] = (15, 15)
plt.axis('off')
plt.imshow(img)
plt.show()

dsp.display(dsp.Audio(f"{basename}.mp3"))